# HouseholdBasketEnv — Baseline Eval (Phase 4.5 / Phase 5)

Run **vanilla Qwen2.5-3B-Instruct** (no fine-tuning) on the held-out seeds for Tasks 1/2/3. We record:

- per-seed total reward
- format-error rate (failed JSON / schema)
- terminal status (`+1.0` / `0.0` / `-0.5`)

Output is `docs/results/baseline.json` — the **must-beat** number for the trained agent (plan §10).

> **Hardware:** Local GPU or Colab T4. The first cell clones the repo and installs deps automatically.

## 1. Environment setup

The cell below clones `HouseholdBasketEnv` (if not already present) and installs all Python dependencies. Works on both local GPU machines and Colab.

In [9]:
import os, sys, pathlib, subprocess
import unsloth
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

CLONE_DIR = pathlib.Path("/app/hackathon/HouseholdBasketEnv")

REPO_ROOT = CLONE_DIR.resolve()
ENV_ROOT  = REPO_ROOT / "household_basket_env"
RESULTS_DIR = REPO_ROOT / "notebooks" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


os.environ["HOUSEHOLD_BASKET_PRODUCTS_PATH"] = str(ENV_ROOT / "data" / "products.json")
os.environ.setdefault("HF_HOME", str(REPO_ROOT / ".hf_cache"))

print("REPO_ROOT   =", REPO_ROOT)
print("ENV_ROOT    =", ENV_ROOT)
print("RESULTS_DIR =", RESULTS_DIR)

REPO_ROOT   = /app/hackathon/HouseholdBasketEnv
ENV_ROOT    = /app/hackathon/HouseholdBasketEnv/household_basket_env
RESULTS_DIR = /app/hackathon/HouseholdBasketEnv/notebooks/results


In [10]:
# !pip -q install --upgrade pip
# !pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip -q install --no-deps "trl<0.20.0" peft accelerate bitsandbytes
# !pip -q install pydantic==2.* fastapi uvicorn
import importlib, sys
sys.path.insert(0, "/app/hackathon/HouseholdBasketEnv")

from household_basket_env.server.environment import HouseholdBasketEnvironment
from household_basket_env.models import BasketAction, BasketObservation
from household_basket_env.server.curriculum import TIER_CONFIGS, load_verified_seeds

env = HouseholdBasketEnvironment()
print("Environment OK. Tiers:", list(TIER_CONFIGS.keys()))

Environment OK. Tiers: [1, 2, 3]


## 2. Load the base model

We use **Qwen/Qwen2.5-3B-Instruct** loaded in 4-bit through Unsloth (matches the training notebook's model so the comparison is apples-to-apples). No LoRA / no fine-tune — this is the "vanilla prompting" baseline.

In [11]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # MUST be before torch import

os.environ["HTTP_PROXY"] = "http://azureproxy.jio.com:8080"  # MUST be before torch import
os.environ["HTTPS_PROXY"] = "http://azureproxy.jio.com:8080"  # MUST be before torch import

import time
import threading
import torch

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_SEQ_LEN = 2048
LOAD_IN_4BIT = True

# ── GPU diagnostics before loading ──────────────────────────────────────────
print("=" * 50)
print("🔍 SYSTEM CHECK")
print("=" * 50)
print(f"PyTorch version      : {torch.__version__}")
print(f"CUDA available       : {torch.cuda.is_available()}")
print(f"CUDA_VISIBLE_DEVICES : {os.environ.get('CUDA_VISIBLE_DEVICES', 'not set')}")
print(f"Visible GPU count    : {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        free, total = torch.cuda.mem_get_info(i)
        print(f"GPU {i}  : {props.name}")
        print(f"        VRAM total : {total/1e9:.1f} GB")
        print(f"        VRAM free  : {free/1e9:.1f} GB")
else:
    print("⚠️  No GPU detected — model will load on CPU (very slow!)")
print("=" * 50)

if torch.cuda.device_count() != 1:
    print("⚠️  WARNING: CUDA_VISIBLE_DEVICES did not take effect!")
    print("   Please restart the kernel and run this cell first before anything else.")
else:
    print("✅ Correctly isolated to 1 GPU")

print(f"Model   : {MODEL_NAME}")
print(f"4bit    : {LOAD_IN_4BIT}")
print(f"Max Seq : {MAX_SEQ_LEN}")
print("=" * 50)

# ── VRAM helpers ─────────────────────────────────────────────────────────────
def print_vram(label):
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            free, total = torch.cuda.mem_get_info(i)
            used = (total - free) / 1e9
            bar = "█" * int(used / total * 20 * 1e9 / 1e9) 
            print(f"      🖥️  GPU {i} VRAM used: {used:.2f} GB / {total/1e9:.1f} GB  [{label}]")
    else:
        print("      ⚠️  No GPU available")

# ── Background GPU monitor ────────────────────────────────────────────────────
stop_monitor = False

def monitor_gpu():
    prev = [0] * torch.cuda.device_count()
    while not stop_monitor:
        for i in range(torch.cuda.device_count()):
            free, total = torch.cuda.mem_get_info(i)
            used = (total - free) / 1e9
            delta = used - prev[i]
            arrow = f"▲ +{delta:.2f}GB" if delta > 0.05 else ("▼" if delta < -0.05 else "─")
            print(f"  [monitor] GPU {i}: {used:.2f} GB used {arrow}")
            prev[i] = used
        time.sleep(3)

monitor_thread = threading.Thread(target=monitor_gpu, daemon=True)
monitor_thread.start()
print("\n🔁 GPU monitor started (updates every 3s)\n")

# ── Load model ────────────────────────────────────────────────────────────────
import unsloth
from unsloth import FastLanguageModel

model, tokenizer = None, None

try:
    print("\n[1/3] Attempting to load via Unsloth...")
    print_vram("before load")
    t0 = time.time()

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=LOAD_IN_4BIT,
        dtype=None,
    )
    print(f"      ✅ Model weights loaded in {time.time()-t0:.1f}s")
    print_vram("after load")

    print("\n[2/3] Applying inference optimizations...")
    t1 = time.time()
    FastLanguageModel.for_inference(model)
    print(f"      ✅ Inference patch done in {time.time()-t1:.1f}s")

    print("\n[3/3] Checking model device placement...")
    p = next(model.parameters())
    print(f"      ✅ Model is on : {p.device}")
    print(f"      ✅ dtype       : {p.dtype}")
    print_vram("final")

    if "cpu" in str(p.device):
        print("      ⚠️  WARNING: Model loaded on CPU! Inference will be very slow.")
    else:
        print("      ✅ Model is on GPU — good to go!")

    print(f"\n✅ Loaded via Unsloth (total: {time.time()-t0:.1f}s)")

except Exception as e:
    print(f"\n❌ Unsloth failed: {e}")
    print("   Falling back to HuggingFace Transformers...\n")

    from transformers import AutoModelForCausalLM, AutoTokenizer

    print("[1/2] Loading tokenizer...")
    t0 = time.time()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    print(f"      ✅ Tokenizer loaded in {time.time()-t0:.1f}s")

    print("\n[2/2] Loading model (float16, device_map=auto)...")
    print_vram("before load")
    t1 = time.time()
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    print(f"      ✅ Model loaded in {time.time()-t1:.1f}s")
    print_vram("after load")

    p = next(model.parameters())
    print(f"      ✅ Model is on : {p.device}")
    print(f"      ✅ dtype       : {p.dtype}")

    if "cpu" in str(p.device):
        print("      ⚠️  WARNING: Model loaded on CPU! Inference will be very slow.")
    else:
        print("      ✅ Model is on GPU — good to go!")

    print(f"\n✅ Loaded via Transformers (total: {time.time()-t1:.1f}s)")

finally:
    stop_monitor = True
    print("\n🔁 GPU monitor stopped")

print("\n" + "=" * 50)
print("🚀 Model ready!")
print("=" * 50)

🔍 SYSTEM CHECK
PyTorch version      : 2.10.0+cu128
CUDA available       : True
CUDA_VISIBLE_DEVICES : 1
Visible GPU count    : 2
GPU 0  : NVIDIA A100 80GB PCIe
        VRAM total : 85.1 GB
        VRAM free  : 25.9 GB
GPU 1  : NVIDIA A100 80GB PCIe
        VRAM total : 85.1 GB
        VRAM free  : 53.0 GB
⚠️  WARNING: CUDA_VISIBLE_DEVICES did not take effect!
   Please restart the kernel and run this cell first before anything else.
Model   : Qwen/Qwen2.5-3B-Instruct
4bit    : True
Max Seq : 2048

🔁 GPU monitor started (updates every 3s)


[1/3] Attempting to load via Unsloth...
  [monitor] GPU 0: 59.16 GB used ▲ +59.16GB
      🖥️  GPU 0 VRAM used: 59.16 GB / 85.1 GB  [before load]
      🖥️  GPU 1 VRAM used: 32.08 GB / 85.1 GB  [before load]
  [monitor] GPU 1: 32.08 GB used ▲ +32.08GB
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0. vLLM: 0.19.1.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 2. Max memory: 79.254 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10

Loading weights: 100%|██████████| 434/434 [00:00<00:00, 813.90it/s]


  [monitor] GPU 0: 61.30 GB used ▲ +2.14GB
  [monitor] GPU 1: 32.10 GB used ─
  [monitor] GPU 0: 61.30 GB used ─
  [monitor] GPU 1: 32.10 GB used ─
  [monitor] GPU 0: 61.30 GB used ─
  [monitor] GPU 1: 32.10 GB used ─
  [monitor] GPU 0: 61.30 GB used ─
  [monitor] GPU 1: 32.10 GB used ─
unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
      ✅ Model weights loaded in 16.2s
      🖥️  GPU 0 VRAM used: 59.46 GB / 85.1 GB  [after load]
      🖥️  GPU 1 VRAM used: 32.08 GB / 85.1 GB  [after load]

[2/3] Applying inference optimizations...
      ✅ Inference patch done in 0.0s

[3/3] Checking model device placement...
      ✅ Model is on : cuda:0
      ✅ dtype       : torch.bfloat16
      🖥️  GPU 0 VRAM used: 59.46 GB / 85.1 GB  [final]
      🖥️  GPU 1 VRAM used: 32.08 GB / 85.1 GB  [final]
      ✅ Model is on GPU — good to go!

✅ Loaded via Unsloth (total: 16.2s)

🔁 GPU monitor stopped

🚀 Model ready!


## 3. Prompt template + agent loop

The system prompt enforces the **`{"product_id": ..., "member_id": ..., "rationale": ...}`** schema (plan §3). Each step the agent sees:

- household summary (members, allergies, caps)
- catalog (product_id, category, meal_type, price, key nutrients)
- basket so far
- budget remaining + steps remaining

In [ ]:
SYSTEM_PROMPT = """You are a careful nutrition-aware shopping assistant for an Indian household. \
At each step you pick exactly ONE product from the catalog and assign it to ONE household member. \
You MUST output ONLY a JSON object with three keys: \"product_id\" (string), \"member_id\" (string), \"rationale\" (1 short sentence). \
Do NOT add any prose, code fences, or extra keys. Respect each member's caps and allergies; spread meal types across breakfast/lunch/dinner/snack/beverage; stay under the budget."""

import json, re
from textwrap import dedent

MAX_CANDS = 50

def render_observation(obs) -> str:
    members = []
    for m in obs.household:
        caps = ", ".join(f"{k}={v:.0f}" for k, v in m.thresholds_cap.items())
        members.append(f"- {m.member_id} | conds={m.conditions} | caps: {caps}")
    cands = []
    for c in obs.candidates[:MAX_CANDS]:
        nuts = c.nutrition_per_100g
        cands.append(
            f"- {c.product_id} | {c.category} | meal={c.meal_type} | INR {c.price_inr:.0f} | "
            f"sugars_g={nuts.get('sugars_g',0):.1f} sodium_mg={nuts.get('sodium_mg',0):.0f} fat_g={nuts.get('fat_g',0):.1f}"
        )
    basket = []
    for t in obs.basket_so_far:
        basket.append(f"- {t.product_id} -> {t.member_id}")
    return dedent(f"""
    Household:
    {chr(10).join(members)}

    Catalog (top {MAX_CANDS}):
    {chr(10).join(cands)}

    Basket so far ({len(basket)}):
    {chr(10).join(basket) if basket else "(empty)"}

    Budget remaining: INR {obs.budget_remaining:.0f}
    Steps used: {obs.step_index} / {obs.max_steps}
    Attempts used: {obs.attempt_index} / {obs.max_steps * 4}

    Now pick ONE product and assign it to ONE member. Output JSON only.
    """).strip()


# --- JSON extraction ---------------------------------------------------------
# Robust to: bare object, ```json fenced block, leading prose, nested objects.
# The previous regex `\{[^{}]*\}` failed on nested braces, which is why
# Task 2/3 baselines hit ~100% parse-fail.
_FENCE_RE = re.compile(r"```(?:json)?\s*(.*?)```", re.S | re.I)


def _strip_fences(s: str) -> str:
    m = _FENCE_RE.search(s)
    return m.group(1).strip() if m else s.strip()


def extract_json(text: str):
    if not text:
        return None
    s = _strip_fences(text)
    # Fast path: whole string is the JSON.
    try:
        return json.loads(s)
    except Exception:
        pass
    # Brace-balance scan: find the first balanced {...} that parses.
    start = s.find("{")
    while start != -1:
        depth = 0
        for i in range(start, len(s)):
            ch = s[i]
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    candidate = s[start:i + 1]
                    try:
                        return json.loads(candidate)
                    except Exception:
                        break
        start = s.find("{", start + 1)
    return None


# --- Generation: single + batched -------------------------------------------
# Decoder-only models need LEFT padding so the new tokens line up at the end
# of every sequence in the batch.
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id


def chat_generate(system: str, user: str, max_new_tokens: int = 96, temperature: float = 0.2) -> str:
    return batched_chat_generate(system, [user], max_new_tokens=max_new_tokens, temperature=temperature)[0]


def batched_chat_generate(system: str, user_prompts: list, max_new_tokens: int = 128, temperature: float = 0.2):
    """Generate completions for many user prompts in one forward pass."""
    if not user_prompts:
        return []
    texts = []
    for u in user_prompts:
        msgs = [{"role": "system", "content": system}, {"role": "user", "content": u}]
        texts.append(tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True
        ))
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LEN - max_new_tokens,
    ).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=temperature > 0.0,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )
    in_len = inputs["input_ids"].shape[-1]
    return [tokenizer.decode(out[i][in_len:], skip_special_tokens=True) for i in range(out.shape[0])]


print("Prompt + generation helpers ready (batched + brace-balance JSON).")


## 4. Run held-out eval per tier

We use the **`held_out_seeds`** lists generated by `seed_verifier`. For each seed we run one full episode, capping retries by the env's attempt-cap.

In [ ]:
from collections import Counter
from time import time

# -----------------------------------------------------------------------------
# QUICK_MODE — flip this for fast iteration on the eval harness itself.
#   True  : 10 seeds/tier, 96 new tokens. Use for sanity / regex / batching.
#   False : 30 seeds/tier, 128 new tokens. Headline baseline number.
# -----------------------------------------------------------------------------
QUICK_MODE = True

if QUICK_MODE:
    MAX_SEEDS_PER_TIER = 10
    MAX_GENERATION_TOKENS = 96
    BATCH_SIZE = 8
else:
    MAX_SEEDS_PER_TIER = 30
    MAX_GENERATION_TOKENS = 128
    BATCH_SIZE = 8

TEMPERATURE = 0.2


def run_episodes_batched(seeds, task_id: int, batch_size: int = 8):
    """Run multiple episodes in parallel; batch the model.generate calls."""
    rows = []
    for batch_start in range(0, len(seeds), batch_size):
        batch_seeds = seeds[batch_start:batch_start + batch_size]
        envs = [HouseholdBasketEnvironment() for _ in batch_seeds]
        obss = [e.reset(seed=int(s), task_id=task_id) for e, s in zip(envs, batch_seeds)]
        totals = [0.0] * len(batch_seeds)
        parse_fails = [0] * len(batch_seeds)
        iter_cap = obss[0].max_steps * 2 + 2

        for _ in range(iter_cap):
            active_idx = [i for i, o in enumerate(obss) if not o.done]
            if not active_idx:
                break
            user_prompts = [render_observation(obss[i]) for i in active_idx]
            outputs = batched_chat_generate(
                SYSTEM_PROMPT,
                user_prompts,
                max_new_tokens=MAX_GENERATION_TOKENS,
                temperature=TEMPERATURE,
            )
            for j, i in enumerate(active_idx):
                payload = extract_json(outputs[j])
                if payload is None:
                    payload = outputs[j]  # let env's parse-check fire P_parse
                    parse_fails[i] += 1
                out = envs[i].apply_raw_action(payload)
                totals[i] += out.reward or 0.0
                obss[i] = out

        for i, s in enumerate(batch_seeds):
            last = obss[i]
            rows.append({
                "seed": int(s),
                "task_id": task_id,
                "total_reward": round(totals[i], 4),
                "parse_fails": parse_fails[i],
                "step_index": last.step_index,
                "attempt_index": last.attempt_index,
                "terminal_reward": last.terminal_reward,
                "basket_size": len(last.basket_so_far),
            })
    return rows


records = {}
for task_id in (1, 2, 3):
    payload = load_verified_seeds(task_id)
    seeds = payload["held_out_seeds"][:MAX_SEEDS_PER_TIER]
    print(f"\n=== Task {task_id} | {len(seeds)} held-out seeds | batch={BATCH_SIZE} ===")
    t0 = time()
    rows = run_episodes_batched(seeds, task_id, batch_size=BATCH_SIZE)
    mean_r = sum(r["total_reward"] for r in rows) / max(1, len(rows))
    print(f"  done in {time()-t0:.0f}s | mean_reward={mean_r:.3f}")
    records[str(task_id)] = rows

print("\nDone. Aggregating ...")


In [7]:
import statistics, json

summary = {}
for task_id, rows in records.items():
    rewards = [r["total_reward"] for r in rows]
    parse_rate = sum(1 for r in rows if r["parse_fails"] > 0) / max(1, len(rows))
    term_dist = Counter(round(r["terminal_reward"], 1) for r in rows)
    summary[task_id] = {
        "n_seeds": len(rows),
        "mean_reward": round(statistics.fmean(rewards), 4),
        "median_reward": round(statistics.median(rewards), 4),
        "stdev_reward": round(statistics.pstdev(rewards), 4) if len(rewards) > 1 else 0.0,
        "parse_failure_rate": round(parse_rate, 4),
        "terminal_reward_distribution": dict(term_dist),
    }

print("\nBASELINE SUMMARY")
print("================")
for k, v in summary.items():
    print(f"Task {k}: mean={v['mean_reward']:.3f} median={v['median_reward']:.3f} parse_fail={v['parse_failure_rate']:.2%} term={v['terminal_reward_distribution']}")

out_path = RESULTS_DIR / "baseline.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump({
        "model": MODEL_NAME,
        "load_in_4bit": LOAD_IN_4BIT,
        "max_seeds_per_tier": MAX_SEEDS_PER_TIER,
        "summary": summary,
        "rows": records,
    }, f, indent=2)
print(f"\nSaved -> {out_path}")


BASELINE SUMMARY
Task 1: mean=1.323 median=1.786 parse_fail=0.00% term={0.3: 20, 0.0: 8, -0.5: 2}
Task 2: mean=-2.439 median=-2.500 parse_fail=96.67% term={0.0: 30}
Task 3: mean=-3.383 median=-3.500 parse_fail=100.00% term={0.0: 30}

Saved -> /app/hackathon/HouseholdBasketEnv/docs/results/baseline.json


### Next steps

- Push `docs/results/baseline.json` to the repo so `eval_and_plots.ipynb` can compare.
- Move on to `training.ipynb` to start GRPO from this baseline.